# LFM2.5-8B-A1B 8-bit τ³-Bench Retail Eval

Liquid released `LFM2.5-8B-A1B`, an on-device MoE model with native tool-calling support and day-one MLX support. This notebook runs the official 40 held-out τ³-Bench retail eval tasks with the 8-bit MLX checkpoint.

The important difference from the Qwen student notebook is the tool-call syntax. Qwen emits XML-style `<tool_call>` blocks. LFM2.5 emits Python-like calls inside `<|tool_call_start|>...<|tool_call_end|>`. The harness uses a Liquid-specific parser in `common/liquid_format.py` and the same τ³ retail simulator/evaluator as the other notebooks.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import sys
import threading

cwd = Path.cwd()
ROOT = cwd if (cwd / "common" / "config.py").exists() else cwd.parent
if not (ROOT / "common" / "config.py").exists():
    raise RuntimeError("Run this notebook from the repo root or a blog folder under the repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common import config as cfg
from common import liquid_format
from common import retail_eval
from common import tau_runtime
from common import user_simulator

paths = cfg.setup_notebook_paths(blog_dir_name="1-distilling-a-0-8b-tool-calling-agent")
ROOT = paths.root
BLOG_DIR = paths.blog_dir
DATA_DIR = paths.data_dir
OUTPUT_DIR = paths.output_dir
ENV_PATH = paths.env_path
DOTENV_LOADED = paths.dotenv_loaded

TAU_BENCH_REPO_REVISION = cfg.TAU_BENCH_REPO_REVISION
MAX_NEW_TOKENS = cfg.MAX_NEW_TOKENS

print("Project root:", ROOT)
print("Loaded .env:", DOTENV_LOADED, "from", ENV_PATH)

Project root: /Users/kargarisaac/codes/personal/distillation-blogs
Loaded .env: True from /Users/kargarisaac/codes/personal/distillation-blogs/.env


## Model Choice

We use the official 8-bit MLX checkpoint because this is an eval notebook: quality matters more than shaving memory to the minimum. If you want to try a smaller memory footprint later, set `LFM_STUDENT_MODEL=LiquidAI/LFM2.5-8B-A1B-MLX-4bit` before running.

In [ ]:
LFM_STUDENT_MODEL = "LiquidAI/LFM2.5-8B-A1B-MLX-8bit"
LFM_EVAL_LIMIT = 40
LFM_EVAL_MAX_STEPS = 100
LFM_EVAL_MAX_ERRORS = 10
LFM_EVAL_SEED = 42
LFM_EVAL_MAX_NEW_TOKENS = MAX_NEW_TOKENS

print("LFM model:", LFM_STUDENT_MODEL)
print("Eval limit:", LFM_EVAL_LIMIT)
print("Max steps:", LFM_EVAL_MAX_STEPS)
print("Max tool errors:", LFM_EVAL_MAX_ERRORS)
print("Max new tokens per agent action:", LFM_EVAL_MAX_NEW_TOKENS)

In [3]:
retail_domain = tau_runtime.load_tau_bench_retail_domain(DATA_DIR, TAU_BENCH_REPO_REVISION)
tau_bench_runtime = retail_domain.runtime
retail_tools = retail_domain.tools
retail_tool_schema_by_name = retail_domain.tool_schema_by_name
retail_policy = retail_domain.policy

retail_tasks = json.loads(retail_domain.paths["tasks"].read_text())
retail_splits = json.loads(retail_domain.paths["splits"].read_text())
retail_tasks_by_id = {task["id"]: task for task in retail_tasks}
retail_eval_tasks = [retail_tasks_by_id[task_id] for task_id in retail_splits["test"]]
retail_eval_task_objects = tau_bench_runtime.get_tasks("test")[:LFM_EVAL_LIMIT]

print("Pinned tau2-bench checkout:", tau_bench_runtime.source_checkout)
print("Retail tools:", len(retail_tools))
print("Official held-out test tasks:", len(retail_eval_tasks))
print("Tasks selected for this run:", len(retail_eval_task_objects))
print("First eval task ids:", [str(task.id) for task in retail_eval_task_objects[:10]])

Pinned tau2-bench checkout: /Users/kargarisaac/codes/personal/distillation-blogs/data/external/tau2-bench
Retail tools: 16
Official held-out test tasks: 40
Tasks selected for this run: 40
First eval task ids: ['5', '9', '12', '17', '18', '26', '27', '32', '33', '36']


In [4]:
# Parser smoke test for the native LFM2.5 tool-call syntax.
probe_text = """
<think>I need to look up the order.</think>
<|tool_call_start|>[get_order_details(order_id='#W1234567')]<|tool_call_end|><|im_end|>
""".strip()
probe_parse = liquid_format.parse_liquid_tool_calls(probe_text)
print("Parse errors:", probe_parse.errors)
print("Parsed calls:", probe_parse.calls)
print("Cleaned natural text:", liquid_format.strip_liquid_generated_special_tokens(probe_text))

Parse errors: []
Parsed calls: [ParsedLiquidToolCall(name='get_order_details', arguments={'order_id': '#W1234567'}, raw_call="get_order_details(order_id='#W1234567')")]
Cleaned natural text: [get_order_details(order_id='#W1234567')]


In [5]:
from mlx_lm import load as mlx_load
from mlx_lm.sample_utils import make_sampler

lfm_model, lfm_tokenizer = mlx_load(LFM_STUDENT_MODEL)
lfm_sampler = make_sampler(temp=0.0, top_p=1.0, top_k=0)
generation_lock = threading.Lock()

print("Loaded LFM model with MLX-LM:", LFM_STUDENT_MODEL)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loaded LFM model with MLX-LM: LiquidAI/LFM2.5-8B-A1B-MLX-8bit


In [6]:
def debug_user_message_from_task(task: dict) -> str:
    instructions = task["user_scenario"]["instructions"]
    return "".join(
        part
        for part in [
            instructions.get("reason_for_call"),
            instructions.get("known_info"),
            instructions.get("unknown_info"),
        ]
        if part
    )

retail_system_message = tau_runtime.retail_agent_system_prompt(retail_policy)
retail_probe_messages = [
    {"role": "system", "content": retail_system_message},
    {"role": "user", "content": debug_user_message_from_task(retail_eval_tasks[0])},
]
retail_first_turn_prompt = lfm_tokenizer.apply_chat_template(
    retail_probe_messages,
    tools=retail_tools,
    tokenize=False,
    add_generation_prompt=True,
)

print("Rendered LFM prompt preview:")
print(retail_first_turn_prompt[:5000])

Rendered LFM prompt preview:
<|startoftext|><|im_start|>system
You are a retail customer-support action policy agent.
Follow the retail policy exactly. Use tools when tool results are needed.
If you call a tool, output only the tool call. If you need user confirmation or the task is complete, answer the user in natural language.

# Retail policy
# Retail agent policy

As a retail agent, you can help users:

- **cancel or modify pending orders**
- **return or exchange delivered orders**
- **modify their default user address**
- **provide information about their own profile, orders, and related products**

At the beginning of the conversation, you have to authenticate the user identity by locating their user id via email, or via name + zip code. This has to be done even when the user already provides the user id.

Once the user has been authenticated, you can provide the user with information about order, product, profile information, e.g. help the user look up order id.

You can only he

In [7]:
# The user simulator is part of the benchmark environment, not the LFM agent.
# Its model is configured in .env and served through the local ChatGPT subscription shim.
user_simulator_runtime = user_simulator.start_tau_bench_user_simulator_from_env(
    existing_shim=globals().get("chatgpt_user_simulator_shim")
)
chatgpt_user_simulator_shim = user_simulator_runtime.shim
TAU_BENCH_USER_SIMULATOR_LLM = user_simulator_runtime.model
TAU_BENCH_USER_SIMULATOR_ARGS = user_simulator_runtime.args

user_simulator_check = retail_eval.check_tau_bench_retail_user_simulator(
    runtime=tau_bench_runtime,
    task=retail_eval_task_objects[0],
    user_simulator_model=TAU_BENCH_USER_SIMULATOR_LLM,
    user_simulator_args=TAU_BENCH_USER_SIMULATOR_ARGS,
)

print("User simulator model:", TAU_BENCH_USER_SIMULATOR_LLM)
print("ChatGPT subscription shim model:", user_simulator_runtime.shim_model)
print("ChatGPT subscription shim base URL:", chatgpt_user_simulator_shim.base_url)
print("User simulator args:", user_simulator.public_user_simulator_args(TAU_BENCH_USER_SIMULATOR_ARGS))
print("First simulated customer message:")
print(user_simulator_check["content"])

User simulator model: openai/gpt-5.4-mini
ChatGPT subscription shim model: gpt-5.4-mini
ChatGPT subscription shim base URL: http://127.0.0.1:59753/v1
User simulator args: {'api_base': 'http://127.0.0.1:59753/v1', 'base_url': 'http://127.0.0.1:59753/v1', 'temperature': 0.0, 'num_retries': 6, 'timeout': 300}
First simulated customer message:
Hi, I’d like to exchange a couple of items from my order.


In [ ]:
TAU_BENCH_MLFLOW_ENABLED = True
TAU_BENCH_MLFLOW_EXPERIMENT_NAME = "distillation-blogs-tau3"
TAU_BENCH_MLFLOW_LOG_FULL_ARTIFACTS = True

user_simulator_slug = cfg.filename_slug(TAU_BENCH_USER_SIMULATOR_LLM)
lfm_slug = cfg.filename_slug(LFM_STUDENT_MODEL)
lfm_eval_output_path = OUTPUT_DIR / f"{lfm_slug}_tau3_bench_retail_test_official_lfm_mlx_eval_{user_simulator_slug}.json"
lfm_eval_trace_dir = OUTPUT_DIR / "local_traces" / f"{lfm_slug}_tau3_bench_retail_test_official_lfm_mlx_eval_{user_simulator_slug}"
lfm_eval_mlflow_run_name = f"{lfm_slug}_tau3_retail_lfm_mlx_eval_{user_simulator_slug}"
lfm_eval_mlflow_config = cfg.MlflowConfig(
    enabled=TAU_BENCH_MLFLOW_ENABLED,
    experiment_name=TAU_BENCH_MLFLOW_EXPERIMENT_NAME,
    log_full_artifacts=TAU_BENCH_MLFLOW_LOG_FULL_ARTIFACTS,
    log_spans=False,
)

lfm_eval_config = cfg.TauBenchRetailEvalConfig(
    dataset_revision=TAU_BENCH_REPO_REVISION,
    student_model_name=LFM_STUDENT_MODEL,
    user_simulator_model=TAU_BENCH_USER_SIMULATOR_LLM,
    user_simulator_args=TAU_BENCH_USER_SIMULATOR_ARGS,
    max_steps=LFM_EVAL_MAX_STEPS,
    max_errors=LFM_EVAL_MAX_ERRORS,
    max_new_tokens=LFM_EVAL_MAX_NEW_TOKENS,
    seed=LFM_EVAL_SEED,
    model_role="mlx_lfm_8bit",
)

lfm_eval_runner = retail_eval.TauBenchRetailMlxLiquidEvalRunner(
    runtime=tau_bench_runtime,
    model=lfm_model,
    tokenizer=lfm_tokenizer,
    tools=retail_tools,
    tool_schema_by_name=retail_tool_schema_by_name,
    sampler=lfm_sampler,
    config=lfm_eval_config,
    trace_dir=lfm_eval_trace_dir,
    generation_lock=generation_lock,
)

print("Official τ³-Bench retail LFM eval")
print("LFM model:", LFM_STUDENT_MODEL)
print("Runtime: MLX-LM")
print("User simulator model:", TAU_BENCH_USER_SIMULATOR_LLM)
print("Held-out retail tasks:", len(retail_eval_task_objects))
print("Task execution: sequential, one benchmark task at a time")
print("Output path:", lfm_eval_output_path)
print("Trace dir:", lfm_eval_trace_dir)
print("MLflow enabled:", lfm_eval_mlflow_config.enabled)
print("MLflow run name:", lfm_eval_mlflow_run_name)

In [9]:
lfm_eval_payload = retail_eval.run_tau_bench_retail_eval_tasks(
    task_objects=retail_eval_task_objects,
    runner=lfm_eval_runner,
    output_path=lfm_eval_output_path,
    print_progress=True,
    show_progress_bar=True,
    quiet_tau2_console=True,
    mlflow_config=lfm_eval_mlflow_config,
    mlflow_run_name=lfm_eval_mlflow_run_name,
    mlflow_tags={"tau3.notebook": "01_1_lfm2_5_8b_a1b_eval", "tau3.runtime": "mlx_lm"},
)

lfm_eval_rows = lfm_eval_payload["task_results"]
lfm_eval_correct = sum(1 for row in lfm_eval_rows if row["is_success"])
lfm_eval_accuracy = lfm_eval_correct / len(lfm_eval_rows) if lfm_eval_rows else 0.0

print("LFM official τ³-Bench retail pass@1 accuracy:", round(lfm_eval_accuracy, 4))
print(f"Correct: {lfm_eval_correct}/{len(lfm_eval_rows)}")
print("Saved aggregate results to:", lfm_eval_output_path)
print("Saved per-task traces to:", lfm_eval_trace_dir)

failure_counter = Counter(
    row["termination_reason"] if not row.get("error") else f"error:{row['error']['type']}"
    for row in lfm_eval_rows
    if not row["is_success"]
)
print("Failure/termination summary:")
for name, count in failure_counter.most_common():
    print(f"  {name}: {count}")

Cached tasks: 0
Pending tasks: 40


mlx_lfm_8bit eval:   0%|          | 0/40 [00:00<?, ?task/s]

🏃 View run liquidai_lfm2_5_8b_a1b_mlx_8bit_tau3_retail_lfm_mlx_eval_openai_gpt_5_4_mini at: http://127.0.0.1:5050/#/experiments/6/runs/993ad1718c344834a8f05830d4c40c4d
🧪 View experiment at: http://127.0.0.1:5050/#/experiments/6
LFM official τ³-Bench retail pass@1 accuracy: 0.2
Correct: 8/40
Saved aggregate results to: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/liquidai_lfm2_5_8b_a1b_mlx_8bit_tau3_bench_retail_test_official_lfm_mlx_eval_openai_gpt_5_4_mini.json
Saved per-task traces to: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/local_traces/liquidai_lfm2_5_8b_a1b_mlx_8bit_tau3_bench_retail_test_official_lfm_mlx_eval_openai_gpt_5_4_mini
Failure/termination summary:
  user_stop: 32


## What This Gives Us

This is not part of the distillation pipeline yet. It is a fresh student baseline for a stronger Apple-friendly local model. If LFM2.5-8B-A1B does well here, it gives us a useful comparison point: instead of only asking whether distillation improves Qwen 0.8B, we can also ask how much of the gap remains against a newer on-device MoE model.